# Eksperimen Retrieval untuk Chatbot SIKA: LPK Star Skills Innovations
**Asah 2026 · Capstone A26-CS127 · Gen AI · Penerapan matkul Information Retrieval**

Bagian *retrieval* pada RAG bertugas mengambil entri Knowledge Base (KB) yang paling relevan sebelum Gemini menyusun jawaban. Notebook ini membandingkan beberapa metode retrieval dengan metrik IR yang sama:

| Metode | Jenis |
|---|---|
| TF-IDF + cosine | IR klasik (vector space model) |
| BM25 (implementasi sendiri) | IR klasik (probabilistik) |
| Dense retrieval (Sentence Transformers) | Semantik (embedding) |
| Hybrid (BM25 + dense, Reciprocal Rank Fusion) | Gabungan |

**Metrik:** Precision@k, Recall@k, MRR, nDCG@k.
**Data:** `knowledge_base.csv` (17 entri) dan `retrieval_eval_queries.csv` (18 query berlabel), ditambah 8 query parafrase di bawah untuk menguji kemampuan semantik.

> **Keterbatasan:** KB dan query dibuat manual oleh tim dari profil resmi LPK. Jumlahnya kecil, jadi angka metrik bersifat indikatif, bukan bukti kuat.

In [ ]:
# Jalankan sekali di Colab, lalu restart runtime jika diminta
%pip install -q sentence-transformers chromadb

In [ ]:
import re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Muat data

In [ ]:
kb = pd.read_csv('knowledge_base.csv')
ev = pd.read_csv('retrieval_eval_queries.csv')
ev['relevant'] = ev['relevant_kb_ids'].str.split(';')
ev['grup'] = 'lugas'

# Query parafrase: sengaja memakai kata yang berbeda dari isi KB, untuk menguji pemahaman makna
para = [
    ('Apakah saya harus merogoh kocek untuk ikut?', ['KB10', 'KB02']),
    ('Tempat belajarnya ada di mana?', ['KB15']),
    ('Orang yang bisa saya hubungi siapa?', ['KB16']),
    ('Setelah selesai belajar, ada yang mencarikan pekerjaan?', ['KB13']),
    ('Berapa orang yang akan diterima tiap gelombang?', ['KB08']),
    ('Umur berapa yang diperbolehkan?', ['KB06', 'KB07']),
    ('Kursus apa saja yang bisa dipilih?', ['KB03']),
    ('Bukti kelulusan apa yang saya dapat?', ['KB11']),
]
ev_para = pd.DataFrame({'query_id': [f'P{i+1:02d}' for i in range(len(para))],
                        'query': [p[0] for p in para], 'relevant': [p[1] for p in para], 'grup': 'parafrase'})
ev_all = pd.concat([ev[['query_id','query','relevant','grup']], ev_para], ignore_index=True)

kb['teks'] = kb['judul'] + '. ' + kb['isi']
doc_ids = kb['id'].tolist()
print(len(kb), 'entri KB |', len(ev_all), 'query evaluasi')
ev_all.groupby('grup').size()

## 2. Preprocessing
Huruf kecil, hapus tanda baca, buang stopword Indonesia yang umum. Preprocessing yang sama dipakai untuk TF-IDF dan BM25. Dense retrieval memakai teks asli karena model embedding sudah menangani bahasa alami.

In [ ]:
STOP = set("""yang dan di ke dari untuk dengan atau ini itu pada adalah ada apa apakah saya kami kita anda
bisa dapat akan juga sudah belum tidak boleh mau ingin dong sih ya kak saja aja nya berapa bagaimana
bagaimanakah kah lah oleh agar sebagai dalam para serta jika kalau yaitu tersebut""".split())

def tokenize(text):
    t = re.sub(r'[^a-z0-9\s]', ' ', str(text).lower())
    return [w for w in t.split() if w not in STOP]

doc_tokens = [tokenize(t) for t in kb['teks']]
print(doc_tokens[2][:12])

## 3. Metode retrieval
Setiap retriever menerima query dan mengembalikan **skor untuk semua dokumen**, lalu diurutkan.

In [ ]:
class TfidfRetriever:
    name = 'TF-IDF + cosine'
    def __init__(self, docs):
        self.vec = TfidfVectorizer(tokenizer=tokenize, lowercase=False, sublinear_tf=True, token_pattern=None)
        self.D = self.vec.fit_transform(docs)
    def scores(self, q):
        return cosine_similarity(self.vec.transform([q]), self.D)[0]

class BM25Retriever:
    """BM25 Okapi, diimplementasikan sendiri."""
    name = 'BM25'
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = tokenized_docs, k1, b
        self.N = len(tokenized_docs)
        self.avgdl = np.mean([len(d) for d in tokenized_docs])
        df = {}
        for d in tokenized_docs:
            for w in set(d):
                df[w] = df.get(w, 0) + 1
        self.idf = {w: math.log((self.N - n + 0.5) / (n + 0.5) + 1) for w, n in df.items()}
        self.tf = [{w: d.count(w) for w in set(d)} for d in tokenized_docs]
    def scores(self, q):
        out = np.zeros(self.N)
        for i, d in enumerate(self.docs):
            dl = len(d)
            for w in tokenize(q):
                if w in self.tf[i]:
                    f = self.tf[i][w]
                    out[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return out

tfidf = TfidfRetriever(kb['teks'])
bm25 = BM25Retriever(doc_tokens)
print(bm25.scores('syarat umur untuk daftar').round(2))

In [ ]:
from sentence_transformers import SentenceTransformer

EMB_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'   # mendukung bahasa Indonesia
encoder = SentenceTransformer(EMB_MODEL)
doc_emb = encoder.encode(kb['teks'].tolist(), normalize_embeddings=True, show_progress_bar=False)

class DenseRetriever:
    name = 'Dense (Sentence Transformers)'
    def __init__(self, encoder, doc_emb):
        self.enc, self.D = encoder, doc_emb
    def scores(self, q):
        qe = self.enc.encode([q], normalize_embeddings=True, show_progress_bar=False)
        return (qe @ self.D.T)[0]

dense = DenseRetriever(encoder, doc_emb)
print('Dimensi embedding:', doc_emb.shape)

In [ ]:
class HybridRetriever:
    """Reciprocal Rank Fusion (RRF) dari BM25 dan dense retrieval."""
    name = 'Hybrid (BM25 + Dense, RRF)'
    def __init__(self, r1, r2, k=60):
        self.r1, self.r2, self.k = r1, r2, k
    def scores(self, q):
        out = np.zeros(len(doc_ids))
        for r in (self.r1, self.r2):
            order = np.argsort(-r.scores(q))
            for rank, idx in enumerate(order, start=1):
                out[idx] += 1.0 / (self.k + rank)
        return out

hybrid = HybridRetriever(bm25, dense)
retrievers = [tfidf, bm25, dense, hybrid]

## 4. Metrik evaluasi (relevansi biner)
- **Precision@k**: porsi dokumen relevan di k hasil teratas
- **Recall@k**: porsi dokumen relevan yang berhasil ditemukan di k teratas
- **MRR**: rata-rata 1/peringkat dokumen relevan pertama
- **nDCG@k**: kualitas urutan, dokumen relevan di peringkat atas bernilai lebih

In [ ]:
def rank_ids(retriever, q):
    return [doc_ids[i] for i in np.argsort(-retriever.scores(q))]

def precision_at_k(ranked, rel, k): return sum(d in rel for d in ranked[:k]) / k
def recall_at_k(ranked, rel, k):    return sum(d in rel for d in ranked[:k]) / len(rel)
def rr(ranked, rel):
    for i, d in enumerate(ranked, start=1):
        if d in rel: return 1.0 / i
    return 0.0
def ndcg_at_k(ranked, rel, k):
    dcg = sum(1.0 / math.log2(i + 1) for i, d in enumerate(ranked[:k], start=1) if d in rel)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
    return dcg / idcg

def evaluate(retriever, data):
    m = {'P@1': [], 'P@3': [], 'Recall@3': [], 'MRR': [], 'nDCG@3': []}
    for _, row in data.iterrows():
        ranked, rel = rank_ids(retriever, row['query']), row['relevant']
        m['P@1'].append(precision_at_k(ranked, rel, 1)); m['P@3'].append(precision_at_k(ranked, rel, 3))
        m['Recall@3'].append(recall_at_k(ranked, rel, 3)); m['MRR'].append(rr(ranked, rel))
        m['nDCG@3'].append(ndcg_at_k(ranked, rel, 3))
    return {k: float(np.mean(v)) for k, v in m.items()}

## 5. Hasil perbandingan

In [ ]:
rows = []
for r in retrievers:
    for grup, data in [('semua', ev_all)] + [(g, d) for g, d in ev_all.groupby('grup')]:
        rows.append({'Metode': r.name, 'Grup query': grup, **evaluate(r, data)})
res = pd.DataFrame(rows)
res.pivot_table(index='Metode', columns='Grup query', values='MRR').round(3).rename_axis(None, axis=1)

In [ ]:
res[res['Grup query'] == 'semua'].drop(columns='Grup query').set_index('Metode').round(3)

In [ ]:
pv = res[res['Grup query'] != 'semua'].pivot_table(index='Metode', columns='Grup query', values='MRR')
pv.plot(kind='barh', figsize=(8, 4), title='MRR per metode dan jenis query')
plt.xlabel('MRR'); plt.tight_layout(); plt.show()

**Cara membaca:** bandingkan kolom `lugas` (query memakai kata yang sama dengan isi KB) dan `parafrase` (kata berbeda, makna sama). Metode leksikal (TF-IDF, BM25) biasanya kuat di `lugas` dan turun di `parafrase`, sedangkan dense retrieval lebih tahan terhadap parafrase. Tulis temuan sebenarnya dari tabel kalian di README, jangan menyalin dugaan ini.

## 6. Analisis kegagalan (metode terbaik berdasarkan MRR)

In [ ]:
best_name = res[res['Grup query'] == 'semua'].sort_values('MRR', ascending=False).iloc[0]['Metode']
best = [r for r in retrievers if r.name == best_name][0]
print('Metode terbaik:', best_name)
fail = []
for _, row in ev_all.iterrows():
    ranked = rank_ids(best, row['query'])
    if ranked[0] not in row['relevant']:
        fail.append({'query': row['query'], 'seharusnya': ';'.join(row['relevant']), 'top-1': ranked[0], 'grup': row['grup']})
pd.DataFrame(fail) if fail else 'Semua query berhasil di peringkat 1'


## 7. Ambang skor untuk alih ke admin
Aturan chatbot: jika skor kemiripan teratas terlalu rendah, anggap informasinya **tidak ada di KB** dan alihkan ke admin (WhatsApp). Uji pada dua kelompok query:
- **Ada di KB:** query evaluasi di atas (seharusnya dijawab otomatis)
- **Tidak ada di KB:** pertanyaan masih seputar LPK tetapi informasinya belum ada (seharusnya dialihkan)

Hanya metode dengan skor kemiripan terbatas (TF-IDF cosine dan dense cosine) yang dipakai. Skor BM25 tidak punya skala tetap.

In [ ]:
tidak_ada = ['Apakah ada beasiswa lanjutan setelah lulus?', 'Bisa cicil biaya seragam?', 'Apakah pelatihan ini bisa online?',
             'Ada diskon untuk peserta kedua?', 'Mau tanya soal asrama', 'Apakah ada kelas akhir pekan?',
             'Bagaimana kalau saya penyandang disabilitas?', 'Boleh bawa laptop sendiri?', 'Apakah ada tempat parkir?',
             'Berapa gaji lulusan setelah bekerja?']

def top1(r, qs): return np.array([r.scores(q).max() for q in qs])

for r in (tfidf, dense):
    s_ada, s_tidak = top1(r, ev_all['query']), top1(r, tidak_ada)
    lo, hi = float(min(s_ada.min(), s_tidak.min())), float(max(s_ada.max(), s_tidak.max()))
    rows = []
    for th in np.linspace(lo, hi, 8):
        rows.append({'ambang': round(float(th), 3),
                     '% query ada di KB dijawab otomatis': round((s_ada >= th).mean() * 100, 1),
                     '% query tidak ada di KB dialihkan': round((s_tidak < th).mean() * 100, 1)})
    print(r.name)
    display(pd.DataFrame(rows))

Pilih ambang yang menjaga sebagian besar query "ada di KB" tetap dijawab otomatis, tetapi mengalihkan mayoritas query "tidak ada di KB". Skor nyata antar metode berbeda skala, jadi tentukan ambang untuk metode yang dipakai di produksi saja. Daftar `tidak_ada` hanya 10 contoh, perbanyak dari pertanyaan pengunjung asli saat uji coba LPK.

## 8. Vector database: ChromaDB
Metode terbaik di atas disimpan ke ChromaDB untuk dipakai pada sistem RAG. Contoh di bawah memakai embedding dense yang sama.

In [ ]:
import chromadb

client = chromadb.Client()
try:
    client.delete_collection('kb_lpk')
except Exception:
    pass
col = client.create_collection(name='kb_lpk', metadata={'hnsw:space': 'cosine'})
col.add(ids=doc_ids,
        documents=kb['teks'].tolist(),
        embeddings=doc_emb.tolist(),
        metadatas=[{'kategori': k, 'status': s} for k, s in zip(kb['kategori'], kb['status_verifikasi'])])

def cari(q, n=3):
    qe = encoder.encode([q], normalize_embeddings=True, show_progress_bar=False)
    r = col.query(query_embeddings=qe.tolist(), n_results=n)
    return pd.DataFrame({'id': r['ids'][0], 'kemiripan': [round(1 - d, 3) for d in r['distances'][0]],
                         'status': [m['status'] for m in r['metadatas'][0]], 'isi': r['documents'][0]})

cari('Syarat umur dan pendidikan untuk daftar apa?')

Kolom `status` (terverifikasi / perlu_konfirmasi) bisa dipakai chatbot: jika entri teratas berstatus `perlu_konfirmasi`, jawaban sebaiknya disertai arahan konfirmasi ke admin.

## Catatan untuk README
- **Tujuan:** membandingkan metode retrieval untuk komponen RAG chatbot.
- **Data:** 17 entri KB dan 26 query berlabel (18 lugas, 8 parafrase), dibuat manual dari profil resmi LPK.
- **Metode:** TF-IDF cosine, BM25 (implementasi sendiri), dense retrieval (Sentence Transformers), hybrid RRF.
- **Metrik:** Precision@k, Recall@k, MRR, nDCG@3.
- **Keterbatasan:** data kecil dan dibuat oleh tim, hasil bersifat indikatif. Query dari pengunjung asli akan memberi gambaran lebih jujur.